# Dataset #3: NBA Player Salaries (Kaggle, 2014-15 to 2021-22)
**Source:** [NBA Players and Team Data - Kaggle](https://www.kaggle.com/datasets/loganlauton/nba-players-and-team-data)

**Original dataset:** NBA Salaries (1990-2023)

**Purpose:** Analyze historical player salaries and inflation-adjusted salaries to investigate how player compensation may relate to NBA offseason trades.

Owner: see the project timeline. *Part of the COGS 108 data-cleaning pipeline. See [README.md](README.md) for how the six datasets fit together.*

**Dataset Overview and Important Variables**

This dataset was obtained from Kaggle's NBA Players and Team Data collection, which gives an overview of historical NBA player salaries over the years. Although the original dataset includes seasons beginning in 1990, our group decided to focus on NBA seasons beginning in 2014, which aligns with our research question because it covers the last decade. The available records extend through the 2021-2022 season. After filtering, our dataset originally contained 5,119 records across five columns.

**Important Variables**

The important variables include the player's name (`playerName`), the year their NBA season begins (`seasonStartYear`), their recorded salary (`salary`), and their inflation-adjusted salary (`inflationAdjSalary`). Both salary variables are measured in U.S. dollars. The inflation-adjusted variable may help us compare compensation across different seasons, though the dataset's inflation reference year and adjustment method should be verified before interpreting these comparisons.

This dataset relates to our research because player salary is one of the variables we are studying. Combined with our other datasets, it lets us examine whether player compensation is associated with the likelihood of being traded during the offseason. By comparing player salaries across seasons and eventually combining them with performance statistics and transaction records, we can examine whether salary information helps predict offseason trades. It also gives us a second salary source to cross-check Dataset #1 against, and it is the only one of our sources with an inflation-adjusted salary.

**Data Quality and Cleaning Decisions**

The first step we took was to filter the dataset to include NBA seasons beginning in 2014 or later. We checked the data types and found that the salary columns were stored as text because of dollar signs and commas. We removed those characters and converted both salary columns into numeric values. All records converted successfully without introducing missing values.

We also checked for missing values and found no empty cells in the filtered dataset, so we did not need to remove or replace any observations because of missing values. However, this does not guarantee that every NBA player or contract is represented in the dataset, and at least one full season appears to be missing (see the next paragraph).

During our investigation, we identified 653 exact duplicate records, all in the 2021-2022 season. The 2021 season appears twice in a row in the file, with identical values in every column, including the per-season row number (`Unnamed: 0`). Because the file is named "1990-2023" but its last season is 2021, the second copy was most likely meant to be the 2022-2023 season and was scraped incorrectly. We removed the extra copies to avoid counting the same player-season multiple times, which reduced our dataset from 5,119 to 4,466 records. This means the 2022-2023 season is effectively missing from this source. We also removed the unnecessary index column and reset the row numbers.

We identified 341 salary records below $100,000, representing approximately 7.6% of our cleaned dataset. Rather than deleting these observations, we created a `lowSalaryFlag` column because unusually low salaries may represent short-term contracts or partial-season compensation. These counts jump from 35 in 2016 to 82 in both 2017 and 2018, which is when the NBA introduced two-way contracts, supporting the partial-season explanation. This matters because many of these players are less established than players on average or larger contracts. We also found no zero or negative salaries.

One limitation is that the dataset does not contain contract expiration dates, player performance, transaction outcomes, or team assignments. Therefore, salary records alone cannot determine whether a player was traded, and these variables will need to come from our other datasets. Joining them will rely on player names, which may be spelled differently across sources (accents, suffixes such as "Jr."). Additionally, this dataset ends with the 2021-2022 season, while some of our other datasets extend to 2024-2025 or later.

## Setup

All file paths come from `nba_cleaning_utils.py`, which points at the `data-files/` folder (`RAW` = `00-raw/`, `INTERIM` = `01-interim/`, `PROCESSED` = `02-processed/`).

In [1]:
import pandas as pd

from nba_cleaning_utils import RAW, INTERIM, PROCESSED

## 3A. Load the data

In [2]:
kaggle_salaries = pd.read_csv(RAW / "NBA Salaries(1990-2023).csv")
kaggle_salaries.head()

,Unnamed: 0,playerName,seasonStartYear,salary,inflationAdjSalary
0,0,Patrick Ewing,1990,"$4,250,000","$9,694,547"
1,1,Hot Rod Williams,1990,"$3,785,000","$8,633,850"
2,2,Hakeem Olajuwon,1990,"$3,175,000","$7,242,397"
3,3,Charles Barkley,1990,"$2,900,000","$6,615,103"
4,4,Chris Mullin,1990,"$2,850,000","$6,501,049"


## 3B. Is the data tidy?

Yes. Each row is one observation (one player's salary for one season), each column is one variable (player name, season start year, salary, inflation-adjusted salary), and the whole table is one type of observational unit. The only issues are formatting ones (salaries stored as text with `$` and `,`, and an extra index column), which we fix in the steps below.

## 3C. Size of the data

In [3]:
#Check the number of rows and columns in the data
print("Dataset size:", kaggle_salaries.shape)

#Find the earliest and latest seasons 
print("First season:", kaggle_salaries["seasonStartYear"].min())
print("Last season:", kaggle_salaries["seasonStartYear"].max())

#View the column names
print("Columns:", kaggle_salaries.columns.tolist())

Dataset size: (15857, 5)
First season: 1990
Last season: 2021
Columns: ['Unnamed: 0', 'playerName', 'seasonStartYear', 'salary', 'inflationAdjSalary']


In [4]:
#Keeping all NBA seasons beginning in 2014 or later
kaggle_filtered = kaggle_salaries[kaggle_salaries["seasonStartYear"] >= 2014].copy()

# Number of rows remaining when filtered
print("filtered dataset size:", kaggle_filtered.shape)

#Confirm the seasons included 
print("First season:", kaggle_filtered["seasonStartYear"].min())
print("Last season:", kaggle_filtered["seasonStartYear"].max())

#First 5 rows check
kaggle_filtered.head()

filtered dataset size: (5119, 5)
First season: 2014
Last season: 2021


,Unnamed: 0,playerName,seasonStartYear,salary,inflationAdjSalary
10738,0,Kobe Bryant,2014,"$23,500,000","$29,215,494"
10739,1,Joe Johnson,2014,"$23,180,790","$28,818,648"
10740,2,Carmelo Anthony,2014,"$22,458,401","$27,920,565"
10741,3,Amare Stoudemire,2014,"$21,896,658","$27,222,199"
10742,4,Dwight Howard,2014,"$21,436,271","$26,649,840"


In [5]:
#Check data types and missing values in dataset
kaggle_filtered.info()

<class 'pandas.DataFrame'>
RangeIndex: 5119 entries, 10738 to 15856
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Unnamed: 0          5119 non-null   int64
 1   playerName          5119 non-null   str  
 2   seasonStartYear     5119 non-null   int64
 3   salary              5119 non-null   str  
 4   inflationAdjSalary  5119 non-null   str  
dtypes: int64(2), str(3)
memory usage: 200.1 KB


## 3D. Missing data

We check for empty cells and for placeholder values (such as a repeated dummy salary), then confirm that converting the salary columns to numbers does not create any new missing values.

In [6]:
#Count missing values in each column
print("Missing values by column:")
print(kaggle_filtered.isna().sum())

#Looking for common missing-value placeholders in salary columns
print("\nMost common salary values:")
print(kaggle_filtered["salary"].value_counts().head(10))

Missing values by column:
Unnamed: 0            0
playerName            0
seasonStartYear       0
salary                0
inflationAdjSalary    0
dtype: int64

Most common salary values:
salary
$77,250        120
$1,517,981      61
$79,568         50
$462,629        46
$53,176         46
$5,000,000      40
$12,000,000     30
$1,312,611      30
$1,782,621      30
$8,000,000      29
Name: count, dtype: int64


In [7]:
#Make another copy so the original filtered data stays unchanged
kaggle_clean = kaggle_filtered.copy()

#Remove "$" and "," then convert salaries into numbers
for col in ["salary", "inflationAdjSalary"]:
    kaggle_clean[col] = pd.to_numeric(
        kaggle_clean[col].str.replace("$", "", regex=False)
                           .str.replace(",", "", regex=False),
        errors="coerce"
    )

#Check whether any salary values could not be converted
print("Missing or invalid values after conversion:")
print(kaggle_clean[["salary", "inflationAdjSalary"]].isna().sum())

#Check new data types
print("\nUpdated data types:")
print(kaggle_clean.dtypes)

Missing or invalid values after conversion:
salary                0
inflationAdjSalary    0
dtype: int64

Updated data types:
Unnamed: 0            int64
playerName              str
seasonStartYear       int64
salary                int64
inflationAdjSalary    int64
dtype: object


## 3E. Suspicious entries: duplicate records

In [8]:
#Check for duplicate rows
print("Exact duplicates:", kaggle_clean.duplicated().sum())

#Check players who appear more than once in the same season 
kaggle_dupes = kaggle_clean[
    kaggle_clean.duplicated(
        subset=["playerName", "seasonStartYear"],
        keep=False
    )
]

print("Rows with repeated player-season combinations:", len(kaggle_dupes))

#Display examples
kaggle_dupes.sort_values(
    ["playerName", "seasonStartYear"]
).head(10)

Exact duplicates: 653
Rows with repeated player-season combinations: 1306


,Unnamed: 0,playerName,seasonStartYear,salary,inflationAdjSalary
14628,77,Aaron Gordon,2021,16409091,17895714
15281,77,Aaron Gordon,2021,16409091,17895714
15117,566,Aaron Henry,2021,223338,243571
15770,566,Aaron Henry,2021,223338,243571
14804,253,Aaron Holiday,2021,3980551,4341179
15457,253,Aaron Holiday,2021,3980551,4341179
14816,265,Aaron Nesmith,2021,3631200,3960177
15469,265,Aaron Nesmith,2021,3631200,3960177
15002,451,Aaron Wiggins,2021,1305761,1424059
15655,451,Aaron Wiggins,2021,1305761,1424059


In [9]:
#Identify the duplicate rows
kaggle_dupe_rows = kaggle_clean[
    kaggle_clean.duplicated()
]

#Count duplicates by season 
print("Duplicate rows by season:")
print(kaggle_dupe_rows["seasonStartYear"].value_counts().sort_index())

Duplicate rows by season:
seasonStartYear
2021    653
Name: count, dtype: int64


In [10]:
#Delete exact duplicate rows
kaggle_clean = kaggle_clean.drop_duplicates().copy()

#Double check new dataset size
print("Dataset size after removing duplicates:", kaggle_clean.shape)

#No duplicate confirmation 
print("Remaining exact duplicates:", kaggle_clean.duplicated().sum())

#Confirm player-season combinations are unique 
print("Repeated player-season records:",
      kaggle_clean.duplicated(
          subset=["playerName", "seasonStartYear"]
      ).sum())

Dataset size after removing duplicates: (4466, 5)
Remaining exact duplicates: 0
Repeated player-season records: 0


## 3E (continued). Outliers: extreme and unusually low salaries

In [11]:
#Summary statistics for player salaries
print("Salary statistics:")
print(kaggle_clean["salary"].describe())

#Check the 5 highest salaries
print("\nFive highest salaries:")
print(
    kaggle_clean.nlargest(5, "salary")[
        ["playerName", "seasonStartYear", "salary"]
    ]
)

#Identify 5 lowest salaries
print("\nFive lowest salaries:")
print(
    kaggle_clean.nsmallest(5, "salary")[
        ["playerName", "seasonStartYear", "salary"]
    ]
)

Salary statistics:
count    4.466000e+03
mean     5.838523e+06
std      7.548043e+06
min      9.266000e+03
25%      1.036018e+06
50%      2.564753e+06
75%      7.705899e+06
max      4.578097e+07
Name: salary, dtype: float64

Five highest salaries:
              playerName  seasonStartYear    salary
14551      Stephen Curry             2021  45780966
14552          John Wall             2021  44310840
14553       James Harden             2021  44310840
14554  Russell Westbrook             2021  44211146
13973      Stephen Curry             2020  43006362

Five lowest salaries:
              playerName  seasonStartYear  salary
11750       John Holland             2015    9266
15203     Melvin Frazier             2021   13294
11749      Dorell Wright             2015   17637
15202  Mfiondu Kabengele             2021   19186
11250         Josh Davis             2014   20000


In [12]:
#Find salaries below $100,000 for further investigation
kaggle_low = kaggle_clean[kaggle_clean["salary"] < 100000]

#Count how many records meet this condition 
print("Salaries below $100,000:", len(kaggle_low))

#Check which seasons contain these salaries
print("\nLow salaries by season:")
print(kaggle_low["seasonStartYear"].value_counts().sort_index())

#Display the first 10 lowest salaries
kaggle_low.sort_values("salary").head(10)

Salaries below $100,000: 341

Low salaries by season:
seasonStartYear
2014    16
2015    16
2016    35
2017    82
2018    82
2019    55
2020    12
2021    43
Name: count, dtype: int64


,Unnamed: 0,playerName,seasonStartYear,salary,inflationAdjSalary
11750,499,John Holland,2015,9266,11505
15203,652,Melvin Frazier,2021,13294,14498
11749,498,Dorell Wright,2015,17637,21899
15202,651,Mfiondu Kabengele,2021,19186,20924
14550,577,Caleb Homesley,2020,20000,22987
11250,512,Josh Davis,2014,20000,24864
12295,544,Dahntay Jones,2016,24022,29532
14549,576,Elijah Bryant,2020,24611,28287
12294,543,Tony Wroten,2016,25000,30735
12293,542,Nick Zeisloft,2016,25000,30735


In [13]:
#Flag unusually low salaries below $100,000 without deleting them
kaggle_clean["lowSalaryFlag"] = kaggle_clean["salary"] < 100000

#Check how many were flagged
print("Low salary records flagged:")
print(kaggle_clean["lowSalaryFlag"].value_counts())

#Check for zero or negative salaries
print("\nZero or negative salaries:")
print((kaggle_clean["salary"] <= 0).sum())

#Display the first five flagged records
kaggle_clean[kaggle_clean["lowSalaryFlag"]].head()

Low salary records flagged:
lowSalaryFlag
False    4125
True      341
Name: count, dtype: int64

Zero or negative salaries:
0


,Unnamed: 0,playerName,seasonStartYear,salary,inflationAdjSalary,lowSalaryFlag
11235,497,Jarvis Varnado,2014,75000,93240,True
11236,498,Drew Gordon,2014,73657,91571,True
11237,499,Jerel McNeal,2014,59686,74202,True
11238,500,David Stockton,2014,59686,74202,True
11239,501,Larry Drew II,2014,53716,66780,True


## 3F. Clean the data

In [14]:
#Remove the extra index column
kaggle_clean = kaggle_clean.drop(columns=["Unnamed: 0"])

#Reset the row numbers after cleaning
kaggle_clean = kaggle_clean.reset_index(drop=True)

#Check the final size and column names 
print("Dataset size:", kaggle_clean.shape)
print("Columns:", kaggle_clean.columns.tolist())

#Display the first five cleaned rows
kaggle_clean.head()

Dataset size: (4466, 5)
Columns: ['playerName', 'seasonStartYear', 'salary', 'inflationAdjSalary', 'lowSalaryFlag']


,playerName,seasonStartYear,salary,inflationAdjSalary,lowSalaryFlag
0,Kobe Bryant,2014,23500000,29215494,False
1,Joe Johnson,2014,23180790,28818648,False
2,Carmelo Anthony,2014,22458401,27920565,False
3,Amare Stoudemire,2014,21896658,27222199,False
4,Dwight Howard,2014,21436271,26649840,False


In [15]:
#Final Validation and Check

#Check final missing values
print("Final missing values:")
print(kaggle_clean.isna().sum())

#Check remaining duplicates
print("\nRemaining duplicate rows:", kaggle_clean.duplicated().sum())

#Check the season range
print("\nSeason range:",
      kaggle_clean["seasonStartYear"].min(),
      "to",
      kaggle_clean["seasonStartYear"].max())

#Check the final dataset size
print("\nFinal dataset size:", kaggle_clean.shape)

Final missing values:
playerName            0
seasonStartYear       0
salary                0
inflationAdjSalary    0
lowSalaryFlag         0
dtype: int64

Remaining duplicate rows: 0

Season range: 2014 to 2021

Final dataset size: (4466, 5)


## 3G. Save the cleaned data

In [16]:
#Save the cleaned NBA salary dataset
kaggle_clean.to_csv(
    PROCESSED / "nba_salaries_2014_2022_clean.csv",
    index=False
)
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [17]:
#Double check CSV readability
kaggle_saved = pd.read_csv(
    PROCESSED / "nba_salaries_2014_2022_clean.csv"
)

#Verify that the saved file matches our cleaned dataset
print("Saved dataset size:", kaggle_saved.shape)
print("Matches cleaned dataset:", kaggle_saved.shape == kaggle_clean.shape)

#Display the first five rows
kaggle_saved.head()

Saved dataset size: (4466, 5)
Matches cleaned dataset: True


,playerName,seasonStartYear,salary,inflationAdjSalary,lowSalaryFlag
0,Kobe Bryant,2014,23500000,29215494,False
1,Joe Johnson,2014,23180790,28818648,False
2,Carmelo Anthony,2014,22458401,27920565,False
3,Amare Stoudemire,2014,21896658,27222199,False
4,Dwight Howard,2014,21436271,26649840,False
